# 03 · Điều phối quá trình huấn luyện trên Kaggle (Train Coordinator)

Mục tiêu của notebook:
1. **Quản lý môi trường Kaggle**: Kiểm tra GPU, dung lượng ổ đĩa `/kaggle/working`, cài đặt thư viện cần thiết.
2. **Chuẩn bị dữ liệu**: Đảm bảo `records.json`, `splits.json` và định dạng YOLO dataset đã sẵn sàng.
3. **Điều phối huấn luyện các mô hình**:
   - `yolov8` / `yolo11` (Detection - Ultralytics)
   - `faster_rcnn` (Detection - Torchvision R50-FPN)
   - `mask_rcnn` (Segmentation - Torchvision R50-FPN)
   - `csrnet` (Density Regression - VGG16 Dilated)
4. **Hỗ trợ Resume & Quản lý Checkpoints**: Phục hồi khi phiên Kaggle hết giờ hoặc ngắt kết nối.
5. **Theo dõi Training Curves trong thời gian thực**: Trực quan hóa Loss, Validation MAE, mAP qua các epochs.

In [ ]:
import os, sys, json, time, shutil, warnings
from pathlib import Path

# ---- Xác định thư mục gốc project: chạy được cả local lẫn Kaggle ----
IN_KAGGLE = Path('/kaggle/working').exists()
if IN_KAGGLE:
    CODE_INPUT = Path('/kaggle/input/fruitcount-code/FruitCount')   # sửa theo tên Kaggle Dataset chứa code
    ROOT = Path('/kaggle/working/FruitCount')
    if not ROOT.exists():
        shutil.copytree(CODE_INPUT, ROOT)
    OUT_BASE = Path('/kaggle/working')
else:
    ROOT = Path.cwd().resolve()
    while ROOT != ROOT.parent and not (ROOT / 'src').is_dir():
        ROOT = ROOT.parent
    OUT_BASE = ROOT / 'outputs'
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image as IPImage

from src.utils import load_yaml, save_yaml, read_json, write_json, load_image, ensure_dir

DS_CFG_PATH = Path(os.environ.get('FRUITCOUNT_DATASET_CFG', ROOT / 'configs/dataset.yaml'))
DS_CFG = load_yaml(DS_CFG_PATH)
PROCESSED = ROOT / DS_CFG.get('processed_dir', 'data/processed')
RUNS = Path(os.environ.get('FRUITCOUNT_RUNS', OUT_BASE / 'runs' if IN_KAGGLE else ROOT / 'runs'))
OUT_DIR = ensure_dir(OUT_BASE / 'notebooks' / '03_train')   # hình + bảng của notebook này

warnings.filterwarnings('ignore', category=UserWarning)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 220)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.spines.top': False, 'axes.spines.right': False})
FAMILY_COLORS = {'detection': '#4C72B0', 'segmentation': '#55A868', 'density': '#C44E52'}


def savefig(fig, name):
    fig.savefig(OUT_DIR / f'{name}.png', dpi=150, bbox_inches='tight')
    plt.show()


class StopExecution(Exception):
    """Dừng 'Run All' một cách êm (không in traceback)."""
    def _render_traceback_(self):
        return []


try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs):
        return x

print(f'Môi trường : {"Kaggle" if IN_KAGGLE else "Local"}')
print(f'ROOT       : {ROOT}')
print(f'PROCESSED  : {PROCESSED}')
print(f'RUNS       : {RUNS}')
print(f'OUT_DIR    : {OUT_DIR}')

## 1. Kiểm tra Môi trường Phần cứng & Kaggle Paths

In [ ]:
import torch

print("Python version   :", sys.version.split()[0])
print("PyTorch version  :", torch.__version__)
print("CUDA Available   :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name      :", torch.cuda.get_device_name(0))
    print("Device Memory    :", f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ Cảnh báo: Đang chạy trên CPU! Quá trình train sẽ rất chậm.")

# Kiểm tra dung lượng đĩa
stat = shutil.disk_usage(ROOT)
print(f"Dung lượng đĩa trống: {stat.free / 1e9:.2f} GB / {stat.total / 1e9:.2f} GB")

## 2. Thiết lập đường dẫn dữ liệu trên Kaggle
Cập nhật `configs/dataset.yaml` trỏ đến Kaggle Input.

In [ ]:
# Tự động tìm dataset MinneApple trong /kaggle/input nếu có
if IN_KAGGLE:
    from pathlib import Path
    cand_images = list(Path('/kaggle/input').rglob('detection/train/images'))
    if cand_images:
        images_dir = cand_images[0]
        masks_dir = images_dir.parent / 'masks'
        DS_CFG['paths']['images_dir'] = str(images_dir)
        DS_CFG['paths']['masks_dir'] = str(masks_dir)
        DS_CFG['processed_dir'] = '/kaggle/working/processed'
        save_yaml(DS_CFG, ROOT / 'configs/dataset.yaml')
        print("Đã cập nhật configs/dataset.yaml theo Kaggle Input:")
        print(DS_CFG['paths'])
    else:
        print("Sử dụng đường dẫn hiện tại trong configs/dataset.yaml:")
        print(DS_CFG['paths'])
else:
    print("Chạy trên Local / Server:")
    print(DS_CFG['paths'])

In [ ]:
import subprocess


def run(cmd, log_file=None):
    """Chạy lệnh con, in log trực tiếp (và ghi ra file nếu có). Trả về mã thoát."""
    cmd = [str(c) for c in cmd]
    print('$', ' '.join(cmd), flush=True)
    f = open(log_file, 'a', encoding='utf-8') if log_file else None
    try:
        p = subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
        for line in p.stdout:
            print(line, end='')
            if f:
                f.write(line)
        p.wait()
    finally:
        if f:
            f.close()
    if p.returncode != 0:
        print(f'⚠️ Lệnh kết thúc với mã {p.returncode}')
    return p.returncode

## 3. Tiền xử lý dữ liệu (Prepare Data)
Chạy script `scripts/prepare_data.py` để sinh `records.json`, `splits.json` và xuất dataset cho YOLO.
Nếu đã có sẵn từ trước, lệnh sẽ thông báo nhanh gọn.

In [ ]:
PROCESSED_DIR = ROOT / DS_CFG.get('processed_dir', 'data/processed')
need_prepare = not (PROCESSED_DIR / 'records.json').exists() or not (PROCESSED_DIR / 'yolo' / 'data.yaml').exists()

if need_prepare:
    print("Bắt đầu tiền xử lý dữ liệu...")
    ret = run(['python', 'scripts/prepare_data.py', '--dataset', 'configs/dataset.yaml'])
    assert ret == 0, "Lỗi khi chạy scripts/prepare_data.py"
else:
    print(f"Dữ liệu đã được chuẩn bị sẵn tại: {PROCESSED_DIR}")

## 4. Cấu hình Huấn luyện
Chọn danh sách các mô hình cần huấn luyện, số epochs và batch size mong muốn.
- `MODELS_TO_TRAIN`: Danh sách mô hình (`yolov8`, `yolo11`, `csrnet`, `faster_rcnn`, `mask_rcnn`).
- `EPOCHS_OVERRIDE`: Số epochs muốn ghi đè (đặt `None` nếu muốn dùng mặc định trong file config của từng model).
- `RESUME`: Đặt `True` nếu phiên làm việc trước bị ngắt và muốn tiếp tục từ `weights/last.pt`.

In [ ]:
# Chọn mô hình muốn chạy trong phiên này
# Gợi ý: Chạy thử nhanh với yolov8 và csrnet trước
MODELS_TO_TRAIN = ['yolov8', 'csrnet']  # Các lựa chọn khác: 'yolo11', 'faster_rcnn', 'mask_rcnn'

EPOCHS_OVERRIDE = None     # Ví dụ đặt 5 hoặc 10 để test thử; None để chạy full config
BATCH_SIZE_OVERRIDE = None # Ví dụ: 4 hoặc 8
RESUME = False             # True nếu muốn tiếp tục huấn luyện từ checkpoint gần nhất

print("Mô hình được chọn :", MODELS_TO_TRAIN)
print("Epochs override   :", EPOCHS_OVERRIDE)
print("Resume mode       :", RESUME)

## 5. Tiến hành Huấn luyện
Vòng lặp gọi `scripts/train.py` cho từng mô hình được chọn.

In [ ]:
train_results = {}

for m in MODELS_TO_TRAIN:
    cfg_file = ROOT / f'configs/models/{m}.yaml'
    if not cfg_file.exists():
        print(f"⚠️ Bỏ qua: Không tìm thấy {cfg_file}")
        continue
    
    print(f"\n{'='*30} BẮT ĐẦU HUẤN LUYỆN: {m} {'='*30}")
    cmd = [
        'python', 'scripts/train.py',
        '--model', str(cfg_file),
        '--out', str(RUNS)
    ]
    if EPOCHS_OVERRIDE is not None:
        cmd.extend(['--epochs', str(EPOCHS_OVERRIDE)])
    if BATCH_SIZE_OVERRIDE is not None:
        cmd.extend(['--batch-size', str(BATCH_SIZE_OVERRIDE)])
    if RESUME:
        cmd.append('--resume')
        
    start_t = time.time()
    ret = run(cmd)
    elapsed = time.time() - start_t
    train_results[m] = {'status': 'Thành công' if ret == 0 else 'Thất bại', 'time_sec': elapsed}
    print(f"{'='*30} KẾT THÚC {m} ({elapsed/60:.1f} phút) {'='*30}\n")

df_train_summary = pd.DataFrame(train_results).T
display(df_train_summary)

## 6. Trực quan hóa Đường cong Huấn luyện (Training Curves)
Đọc file `history.json` / `history.csv` hoặc hình ảnh `training_curves.png` được lưu tại mỗi thư mục `runs/<model_name>/`.

In [ ]:
for m in MODELS_TO_TRAIN:
    run_dir = RUNS / m
    curve_img = run_dir / 'training_curves.png'
    history_json = run_dir / 'history.json'
    
    print(f"--- Kết quả huấn luyện: {m} ---")
    if history_json.exists():
        hist = read_json(history_json)
        df_hist = pd.DataFrame(hist)
        print(f"Đã huấn luyện {len(df_hist)} epochs.")
        if 'val_mae' in df_hist.columns:
            best_mae = df_hist['val_mae'].min()
            best_ep = df_hist.loc[df_hist['val_mae'].idxmin(), 'epoch']
            print(f"Val MAE tốt nhất: {best_mae:.3f} (tại epoch {best_ep})")
        display(df_hist.tail(5))
        
        # Vẽ biểu đồ loss và val_mae
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        loss_cols = [c for c in df_hist.columns if 'loss' in c]
        for c in loss_cols:
            axes[0].plot(df_hist['epoch'], df_hist[c], label=c)
        axes[0].set(title=f"Loss - {m}", xlabel="Epoch", ylabel="Loss")
        axes[0].legend()
        
        if 'val_mae' in df_hist.columns:
            axes[1].plot(df_hist['epoch'], df_hist['val_mae'], 'r-o', label='Val MAE')
            axes[1].set(title=f"Validation MAE - {m}", xlabel="Epoch", ylabel="MAE")
            axes[1].legend()
        plt.tight_layout()
        savefig(fig, f"train_curve_{m}")
        
    elif curve_img.exists():
        display(IPImage(filename=str(curve_img)))
    else:
        # Đối với YOLO, Ultralytics lưu tại results.png hoặc results.csv
        yolo_results = run_dir / 'results.csv'
        if yolo_results.exists():
            df_yolo = pd.read_csv(yolo_results)
            df_yolo.columns = [c.strip() for c in df_yolo.columns]
            print("YOLO results.csv:")
            display(df_yolo.tail(5))

## 7. Tổng kết & Bước tiếp theo
- Các checkpoint mô hình tốt nhất đã được lưu tại: `runs/<model>/weights/best.pt`.
- Bước tiếp theo: Mở `04_evaluate.ipynb` để đánh giá chi tiết trên tập test, đo FPS và phân tích sai số theo mật độ quả.